In [ ]:
import jax
import jax.numpy as jnp
import optax
from sklearn.model_selection import train_test_split
import tensorflow_datasets as tfds

# Cargar y preparar datos
ds = tfds.load('fashion_mnist', split='train', as_supervised=True)
ds = list(ds.take(1000))  # Solo 1000 ejemplos para velocidad
X = jnp.array([x.numpy().astype(jnp.float32).reshape(-1) / 255.0 for x, _ in ds])
y = jnp.array([y.numpy() for _, y in ds])

# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Inicializar pesos manualmente
def init_params(key, input_size=784, hidden_size=128, output_size=10):
    k1, k2 = jax.random.split(key)
    W1 = jax.random.normal(k1, (input_size, hidden_size)) * 0.01
    b1 = jnp.zeros(hidden_size)
    W2 = jax.random.normal(k2, (hidden_size, output_size)) * 0.01
    b2 = jnp.zeros(output_size)
    return W1, b1, W2, b2

# Función de propagación hacia adelante
def forward(params, x):
    W1, b1, W2, b2 = params
    z1 = jnp.dot(x, W1) + b1
    a1 = jnp.maximum(0, z1)  # ReLU
    z2 = jnp.dot(a1, W2) + b2
    return z2  # logits

# Función de pérdida
def loss_fn(params, x, y):
    logits = forward(params, x)
    return optax.softmax_cross_entropy_with_integer_labels(logits, y).mean()

# Inicializar
key = jax.random.PRNGKey(0)
params = init_params(key)

# Optimizador
opt = optax.adam(1e-3)
opt_state = opt.init(params)

# Entrenamiento
@jax.jit
def train_step(params, opt_state, x, y):
    grads = jax.grad(loss_fn)(params, x, y)
    updates, opt_state = opt.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state

# Entrenar por 10 épocas
for epoch in range(10):
    params, opt_state = train_step(params, opt_state, X_train, y_train)
    loss = loss_fn(params, X_train, y_train)
    print(f"Epoch {epoch + 1}, Loss: {loss:.4f}")